In [37]:
from pathlib import Path
import numpy as np
import pandas as pd

# Folder that contains rnn.ipynb (i.e., Project3/Implementations/)
nb_dir = Path.cwd()

data_dir = (nb_dir / ".." / "processed_player_value").resolve()

npz_path = data_dir / "cumlag_rnn_dataset.npz"
csv_path = data_dir / "meta.csv"

print("Notebook dir:", nb_dir)
print("Data dir:", data_dir)
print("NPZ exists:", npz_path.exists(), npz_path)
print("CSV exists:", csv_path.exists(), csv_path)

dataset = np.load(npz_path, allow_pickle=True)
meta = pd.read_csv(csv_path)

print("NPZ keys:", list(dataset.keys()))
print("meta shape:", meta.shape)
meta.head()


Notebook dir: /Users/bror/Documents/GitHub/FYSSTK3155/PROJECT 3/Code/Implementations
Data dir: /Users/bror/Documents/GitHub/FYSSTK3155/PROJECT 3/Code/processed_player_value
NPZ exists: True /Users/bror/Documents/GitHub/FYSSTK3155/PROJECT 3/Code/processed_player_value/cumlag_rnn_dataset.npz
CSV exists: True /Users/bror/Documents/GitHub/FYSSTK3155/PROJECT 3/Code/processed_player_value/meta.csv
NPZ keys: ['X_seq', 'X_static', 'y']
meta shape: (278558, 2)


,player_id,valuation_date
0,10,2013-01-14
1,10,2013-06-19
2,10,2014-01-07
3,10,2014-07-07
4,10,2015-01-07


In [38]:
import pandas as pd

meta = pd.read_csv("../processed_player_value/meta.csv")
meta["valuation_date"] = pd.to_datetime(meta["valuation_date"])

sizes = meta.groupby("player_id").size()

print("Players:", sizes.shape[0])
print("Min rows per player:", sizes.min())
print("Median rows per player:", sizes.median())
print("Max rows per player:", sizes.max())

# Time gaps (days)
meta = meta.sort_values(["player_id", "valuation_date"])
deltas = meta.groupby("player_id")["valuation_date"].diff().dt.days

print("H (days) summary:")
print(deltas.describe())


Players: 22694
Min rows per player: 1
Median rows per player: 11.0
Max rows per player: 41
H (days) summary:
count    255864.000000
mean        168.861489
std         105.527146
min           1.000000
25%         114.000000
50%         161.000000
75%         196.000000
max        3419.000000
Name: valuation_date, dtype: float64


### Pytorch GRU implementation (VIBE)

In [39]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


In [40]:
data_dir = Path("../processed_player_value")

d = np.load(data_dir / "cumlag_rnn_dataset.npz")
X_seq = d["X_seq"]          # (N, 20, 6)
X_static = d["X_static"]    # (N, 12)
y = d["y"]                  # log-market-value

meta = pd.read_csv(data_dir / "meta.csv")
meta["valuation_date"] = pd.to_datetime(meta["valuation_date"])

# Sort by player and time
order = np.lexsort((meta["valuation_date"].values, meta["player_id"].values))
meta = meta.iloc[order].reset_index(drop=True)

X_seq = X_seq[order]
X_static = X_static[order]
y = y[order]

# Build next-step pairs
same_player = meta["player_id"].values[1:] == meta["player_id"].values[:-1]
idx_in = np.where(same_player)[0]
idx_out = idx_in + 1

X_seq = X_seq[idx_in]
X_static = X_static[idx_in]
y_next = y[idx_out]

# Add H (days) as static feature
H_days = (
    meta["valuation_date"].values[idx_out]
    - meta["valuation_date"].values[idx_in]
).astype("timedelta64[D]").astype(np.float32).reshape(-1, 1)

X_static = np.concatenate([X_static, H_days], axis=1)

target_dates = meta["valuation_date"].values[idx_out]

print("Final samples:", len(y_next))
print("Seq shape:", X_seq.shape)
print("Static shape:", X_static.shape)


Final samples: 255864
Seq shape: (255864, 20, 6)
Static shape: (255864, 13)


In [51]:
# Look at one random sample
i = np.random.randint(0, X_seq.shape[0])

print("Sequence shape:", X_seq[i].shape)
print("Sequence (first 5 timesteps):")
print(X_seq[i][:5])


Sequence shape: (20, 6)
Sequence (first 5 timesteps):
[[1. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0.]
 [1. 0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0.]]


In [41]:
import numpy as np

def nan_report(name, a):
    a = np.asarray(a)
    finite = np.isfinite(a)
    print(f"{name}:")
    print("  shape:", a.shape)
    print("  finite %:", finite.mean() * 100)
    print("  nan count:", np.isnan(a).sum())
    print("  inf count:", np.isinf(a).sum())
    if finite.any():
        print("  min:", a[finite].min(), "max:", a[finite].max())
    print()

nan_report("X_seq", X_seq)
nan_report("X_static", X_static)
nan_report("y_next", y_next)


X_seq:
  shape: (255864, 20, 6)
  finite %: 100.0
  nan count: 0
  inf count: 0
  min: 0.0 max: 6.0

X_static:
  shape: (255864, 13)
  finite %: 99.91573047219798
  nan count: 2803
  inf count: 0
  min: 0.0 max: 3419.0

y_next:
  shape: (255864,)
  finite %: 100.0
  nan count: 0
  inf count: 0
  min: 9.210441 max: 19.113829



In [42]:
good = (
    np.isfinite(X_seq).all(axis=(1,2)) &
    np.isfinite(X_static).all(axis=1) &
    np.isfinite(y_next)
)

print("Bad rows:", (~good).sum())
print("Keeping:", good.sum(), "/", len(good))

X_seq = X_seq[good]
X_static = X_static[good]
y_next = y_next[good]
target_dates = target_dates[good]


Bad rows: 2803
Keeping: 253061 / 255864


In [43]:
train_end = pd.Timestamp("2021-12-31")
val_end = pd.Timestamp("2023-12-31")

train_idx = np.where(target_dates <= train_end)[0]
val_idx = np.where((target_dates > train_end) & (target_dates <= val_end))[0]
test_idx = np.where(target_dates > val_end)[0]

print("Train / Val / Test:",
      len(train_idx), len(val_idx), len(test_idx))


Train / Val / Test: 176627 59024 17410


In [44]:
class ValuationDataset(Dataset):
    def __init__(self, X_seq, X_static, y):
        self.X_seq = torch.tensor(X_seq, dtype=torch.float32)
        self.X_static = torch.tensor(X_static, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.X_static[idx], self.y[idx]


In [45]:
class GRUModel(nn.Module):
    def __init__(self, seq_dim, static_dim, hidden_dim=1024):
        super().__init__()
        self.gru = nn.GRU(seq_dim, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim + static_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_seq, x_static):
        _, h = self.gru(x_seq)      # (1, B, H)
        h = h.squeeze(0)            # (B, H)
        x = torch.cat([h, x_static], dim=1)
        return self.fc(x)


In [46]:
def run_epoch(model, loader, optimizer=None, device="cpu"):
    loss_fn = nn.MSELoss()
    total = 0.0
    n = 0

    train = optimizer is not None
    model.train(train)

    for xs, xst, y in loader:
        xs, xst, y = xs.to(device), xst.to(device), y.to(device)
        pred = model(xs, xst)
        loss = loss_fn(pred, y)

        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        total += loss.item() * y.size(0)
        n += y.size(0)

    return total / n


In [47]:
device = "cuda" if torch.cuda.is_available() else "cpu"

train_ds = ValuationDataset(X_seq[train_idx], X_static[train_idx], y_next[train_idx])
val_ds   = ValuationDataset(X_seq[val_idx],   X_static[val_idx],   y_next[val_idx])
test_ds  = ValuationDataset(X_seq[test_idx],  X_static[test_idx],  y_next[test_idx])

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=256)
test_loader  = DataLoader(test_ds, batch_size=256)

model = GRUModel(seq_dim=6, static_dim=X_static.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)


best_val = float("inf")
best_state = None

for epoch in range(1, 11):
    train_loss = run_epoch(model, train_loader, optimizer, device)
    val_loss = run_epoch(model, val_loader, None, device)

    print(f"Epoch {epoch:02d} | "
          f"train MSE: {train_loss:.4f} | "
          f"val MSE: {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        best_state = model.state_dict()

# Load best model safely
if best_state is not None:
    model.load_state_dict(best_state)
else:
    print("WARNING: No best model saved, using last epoch weights")

test_loss = run_epoch(model, test_loader, None, device)
print("Test MSE (log scale):", test_loss)


Epoch 01 | train MSE: 3.5879 | val MSE: 1.7845
Epoch 02 | train MSE: 1.3865 | val MSE: 1.4243
Epoch 03 | train MSE: 1.2434 | val MSE: 1.2837
Epoch 04 | train MSE: 1.1926 | val MSE: 1.2485
Epoch 05 | train MSE: 1.1826 | val MSE: 1.2158
Epoch 06 | train MSE: 1.1695 | val MSE: 1.2177
Epoch 07 | train MSE: 1.1680 | val MSE: 1.1972
Epoch 08 | train MSE: 1.1601 | val MSE: 1.1899
Epoch 09 | train MSE: 1.1624 | val MSE: 1.1934
Epoch 10 | train MSE: 1.1566 | val MSE: 1.2017
Test MSE (log scale): 1.290803489049641


### LTSM model (Vibecoding)

In [48]:
class LSTMModel(nn.Module):
    def __init__(self, seq_dim, static_dim, hidden_dim=1024):
        super().__init__()
        self.lstm = nn.LSTM(seq_dim, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim + static_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_seq, x_static):
        _, (h, c) = self.lstm(x_seq)   # h: (1, B, H)
        h = h.squeeze(0)              # (B, H)
        x = torch.cat([h, x_static], dim=1)
        return self.fc(x)


In [49]:
seq_dim = 6
static_dim = train_loader.dataset[0][1].shape[0] 

model = LSTMModel(seq_dim=seq_dim, static_dim=static_dim, hidden_dim=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_val = float("inf")
best_state = None

for epoch in range(1, 11):
    train_loss = run_epoch(model, train_loader, optimizer, device)
    val_loss = run_epoch(model, val_loader, None, device)

    print(f"Epoch {epoch:02d} | train MSE: {train_loss:.4f} | val MSE: {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

print("Best val MSE:", best_val)

test_loss = run_epoch(model, test_loader, None, device)
print("Test MSE (log scale):", test_loss)


Epoch 01 | train MSE: 2.5101 | val MSE: 1.6316
Epoch 02 | train MSE: 1.3550 | val MSE: 1.3813
Epoch 03 | train MSE: 1.2438 | val MSE: 1.2740
Epoch 04 | train MSE: 1.2012 | val MSE: 1.3178
Epoch 05 | train MSE: 1.1899 | val MSE: 1.2349
Epoch 06 | train MSE: 1.1773 | val MSE: 1.2387
Epoch 07 | train MSE: 1.1629 | val MSE: 1.2358
Epoch 08 | train MSE: 1.1440 | val MSE: 1.2080
Epoch 09 | train MSE: 1.1327 | val MSE: 1.2057
Epoch 10 | train MSE: 1.1215 | val MSE: 1.2491
Best val MSE: 1.2056758171434678
Test MSE (log scale): 1.1926791683002014
